In [2]:

import pandas as pd

df = pd.read_csv('/content/dirty_cafe_sales.csv')

df.head()


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2,4,Credit Card,Takeaway,08-09-2023
1,TXN_4977031,Cake,4,3,12,Cash,In-store,16-05-2023
2,TXN_4271903,Cookie,4,1,ERROR,Credit Card,In-store,19-07-2023
3,TXN_7034554,Salad,2,5,10,UNKNOWN,UNKNOWN,27-04-2023
4,TXN_3160411,Coffee,2,2,4,Digital Wallet,In-store,11-06-2023


In [3]:

print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types and Non-Null Counts:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nUnique Values in Each Column:")
for col in df.columns:
    print(f"\n{col}:")
    print(df[col].value_counts().head(10))


Dataset Shape:
(10000, 8)

Column Names:
['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']

Data Types and Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB

Missing Values:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Tra

In [4]:

# Keep an unchanged copy of the original dataset
df_original = df.copy()

# Create a working copy for cleaning
df_clean = df.copy()

print("Original shape:", df_original.shape)
print("Working copy shape:", df_clean.shape)


Original shape: (10000, 8)
Working copy shape: (10000, 8)


In [5]:

import numpy as np

df_clean = df_clean.replace(
    to_replace=r'^\s*(ERROR|UNKNOWN)\s*$',
    value=np.nan,
    regex=True
)

print("Missing values after cleaning ERROR/UNKNOWN:")
print(df_clean.isnull().sum())


Missing values after cleaning ERROR/UNKNOWN:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


In [6]:

numeric_cols = ['Quantity', 'Price Per Unit', 'Total Spent']

for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print("Data types:")
print(df_clean[numeric_cols].dtypes)

print("\nMissing numeric values:")
print(df_clean[numeric_cols].isnull().sum())



Data types:
Quantity          float64
Price Per Unit    float64
Total Spent       float64
dtype: object

Missing numeric values:
Quantity          479
Price Per Unit    533
Total Spent       502
dtype: int64


In [7]:

# Convert Transaction Date into proper datetime format
df_clean['Transaction Date'] = pd.to_datetime(
    df_clean['Transaction Date'],
    format='%d-%m-%Y',
    errors='coerce'
)

print("Date data type:")
print(df_clean['Transaction Date'].dtype)

print("\nMissing dates after conversion:")
print(df_clean['Transaction Date'].isnull().sum())

print("\nSample dates:")
print(df_clean['Transaction Date'].head())


Date data type:
datetime64[ns]

Missing dates after conversion:
460

Sample dates:
0   2023-09-08
1   2023-05-16
2   2023-07-19
3   2023-04-27
4   2023-06-11
Name: Transaction Date, dtype: datetime64[ns]


In [8]:

# Check missing values in all columns
print("Missing values:")
print(df_clean.isnull().sum())

# Check whether Total Spent matches Quantity × Price Per Unit
check = df_clean.dropna(
    subset=['Quantity', 'Price Per Unit', 'Total Spent']
).copy()

check['Calculated Total'] = (
    check['Quantity'] * check['Price Per Unit']
)

check['Total Matches'] = np.isclose(
    check['Total Spent'],
    check['Calculated Total']
)

print("\nRows checked:", len(check))
print("Matching totals:", check['Total Matches'].sum())
print("Non-matching totals:", (~check['Total Matches']).sum())


Missing values:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

Rows checked: 8544
Matching totals: 8544
Non-matching totals: 0


In [9]:

# Keep rows that have the essential sales information
df_analysis = df_clean.dropna(
    subset=[
        'Item',
        'Quantity',
        'Price Per Unit',
        'Total Spent',
        'Transaction Date'
    ]
).copy()

print("Original rows:", len(df_clean))
print("Rows retained:", len(df_analysis))
print("Rows removed:", len(df_clean) - len(df_analysis))

print("\nRemaining missing values:")
print(df_analysis.isnull().sum())


Original rows: 10000
Rows retained: 7382
Rows removed: 2618

Remaining missing values:
Transaction ID         0
Item                   0
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      2323
Location            2931
Transaction Date       0
dtype: int64


In [10]:

print("Final dataset shape:", df_analysis.shape)

print("\nData types:")
print(df_analysis.dtypes)

print("\nMissing values:")
print(df_analysis.isnull().sum())

print("\nDuplicate rows:")
print(df_analysis.duplicated().sum())

print("\nFirst 5 cleaned rows:")
display(df_analysis.head())


Final dataset shape: (7382, 8)

Data types:
Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object

Missing values:
Transaction ID         0
Item                   0
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      2323
Location            2931
Transaction Date       0
dtype: int64

Duplicate rows:
0

First 5 cleaned rows:


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
3,TXN_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,NaN,2023-03-31


In [11]:

df_analysis.to_csv('/content/cleaned_cafe_sales.csv', index=False)

print("Cleaned CSV saved successfully!")


Cleaned CSV saved successfully!
